In [33]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

class SparkConfig :
    def creat_sparksession() :
        spark = SparkSession.builder.config("spark.driver.memory" , "8g") \
                            .appName("clean data") \
                            .master("local[*]") \
                            .getOrCreate()
        spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", "file:///")
        return spark

In [34]:
spark = SparkConfig.creat_sparksession()

In [35]:
class DataLoader :
    def __init__(selt) :
        selt.spark = SparkConfig.create_sparksession() 
    def clean_works() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            works = spark.read.format("parquet").load(file) 
            
            works = works.withColumn("cover_id" , col("cover_id").cast("long"))
            for c in works.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in works.columns :
                    works = works.withColumn(c , lit(None).cast("string"))
            list_file.append(works)
        works = reduce(lambda a,b : a.union(b) , list_file)
        return works

    def clean_editions() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/editions/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            editions = spark.read.format("parquet").load(file) 
            if "number_of_pages" in editions.columns :
                editions = editions.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            for c in editions.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in editions.columns :
                    editions = editions.withColumn(c , lit(None).cast("string"))
            list_file.append(editions)
        editions = reduce(lambda a,b : a.union(b) , list_file)
        editions.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/editions/")
        return editions 
    
    def clean_works() :        
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/authors/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        for file in files :
            authors = spark.read.format("parquet").load(file) 
            if "number_of_pages" in authors.columns :
                authors = authors.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            list_file.append(authors)
        authors = reduce(lambda a,b : a.unionByName(b , allowMissingColumns = True) , list_file)
        authors.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/authors/")
        return authors

In [36]:
spark

In [37]:
works = spark.read.parquet("data/works.parquet")

In [38]:
editions = spark.read.parquet("data/editions.parquet")

In [39]:
authors = spark.read.parquet("data/authors.parquet")

In [40]:
def work_clean(works) :
    print("\n" + "="*60 )
    print("====== Starting Clean works ======")
    print("="*60 )
    # Chọn các cột cần để phân tích
    works = works.select("key", "title", "first_publish_year", "edition_count", "subject", "authors" )
    print("====== Missing Value in works ======")
    works.select([count(when (col(c).isNull() , c)).alias(c) for c in works.columns ]).show()
    # Làm sạch cột key và first_pubnlish_year 
    work_key = works.withColumn("key" , regexp_extract(col("key") , r"[A-Z]+\d+[A-Z]", 0)) \
                    .withColumnRenamed("key", "work_id") \
                    .withColumn("first_publish_year" , col("first_publish_year").cast("int")) # chuyển đổi kiểu dữ liệu cột first_publish_year
    # Xử lý giá trị missing value bằng cách lấy trung vị
    median = work_key.approxQuantile("first_publish_year", [0.5], 0.01)[0]
    work_key = work_key.fillna({"first_publish_year" : int(median)})
    # Explode cột subject 
    work_explode_sub = work_key.withColumn("subject" , regexp_replace("subject" , r"[\]\[]", "")) \
                            .withColumn("subject", explode(split("subject" , ",")))
    # Làm sạch cột subject
    work_explode_sub = work_explode_sub.withColumn("subject", regexp_replace("subject", r'form:|genre:|"', ""))
    # Làm sạch bảng work_explode_sub
    work_explode_sub = work_explode_sub.filter(col("subject").isNotNull() & ~col("subject").rlike("\\\\|[0-9]"))
    # Xóa bỏ khoảng trắng, lọc các dữ liệu sạch
    work_explode_sub = work_explode_sub.orderBy("subject") \
            .withColumn("subject", trim(col("subject"))) \
            .filter(~col("subject").rlike("&|.com$|@|^:|^\\.") & trim(col("subject") != "") ) \
            .withColumn("subject", regexp_replace("subject" ,"'|\\*|-" , "")) 
    # Làm sạch bảng work_explode_sub
    work_explode_sub = work_explode_sub.filter(col("subject").isNotNull() & ~col("subject").rlike("\\\\|[0-9]"))
    # Xóa bỏ khoảng trắng, lọc các dữ liệu sạch
    work_explode_sub = work_explode_sub.orderBy("subject") \
            .withColumn("subject", trim(col("subject"))) \
            .filter(~col("subject").rlike("&|.com$|@|^:|^\\.") & trim(col("subject") != "") ) \
            .withColumn("subject", regexp_replace("subject" ,"'|\\*|-" , "")) 
    work_explode_sub.show(5)
    print("====== Clean works sucessfully ======")
    return work_explode_sub

In [41]:
work_clean = work_clean(works)


====== Starting Clean works ======
====== Missing Value in works ======
+---+-----+------------------+-------------+-------+-------+
|key|title|first_publish_year|edition_count|subject|authors|
+---+-----+------------------+-------------+-------+-------+
|  0|    0|               145|            0|      0|      0|
+---+-----+------------------+-------------+-------+-------+



+----------+-------------------+------------------+-------------+--------------------+--------------------+
|   work_id|              title|first_publish_year|edition_count|             subject|             authors|
+----------+-------------------+------------------+-------------+--------------------+--------------------+
|OL1652426W|Elizabeth and Essex|              1928|           41|             Earl of|[{"key": "/author...|
|  OL19870W|    The Jungle Book|              1893|          536|             Fiction|[{"key": "/author...|
|OL5361840W|         Henry VIII|              1968|           17|     King of England|[{"key": "/author...|
|  OL54797W|       The Notebook|              1996|           53|North Carolina  F...|[{"key": "/author...|
|OL1652426W|Elizabeth and Essex|              1928|           41|    Queen of England|[{"key": "/author...|
+----------+-------------------+------------------+-------------+--------------------+--------------------+
only showing top 5 rows
====

In [42]:
def dim_work(work_clean):
    print("\n" + "="*60 )
    print("====== Create dimension work  ======")
    print("="*60 )
    # Tạo ra bảng dim_work 
    dim_work = work_clean.select("work_id", 
                                "title", 
                                "first_publish_year", 
                                "edition_count").dropDuplicates()
    dim_work.show(5)
    return dim_work

In [43]:
dim_work = dim_work(work_clean)


====== Create dimension work  ======


+----------+--------------------+------------------+-------------+
|   work_id|               title|first_publish_year|edition_count|
+----------+--------------------+------------------+-------------+
|OL5735363W|    The Hunger Games|              2008|          142|
|OL1227093W|             Matisse|              1943|           60|
| OL272171W|The Shakespearian...|              1932|           26|
|OL2899874W|  L' Orlando furioso|              1545|          167|
|  OL45793W|Charlie and the G...|              1972|          113|
+----------+--------------------+------------------+-------------+
only showing top 5 rows


In [44]:
def dim_subject(work_clean) :
    print("\n" + "="*60 )
    print("====== Create dimension subject  ======")
    print("="*60 )
    # Tạo bảng dim_subject
    dim_subject = work_clean.select("subject").dropDuplicates()
    dim_subject = dim_subject.distinct()
    dim_subject.count()
    # Gen Id cho bảng dim_subject sử dụng window function
    dim_subject = dim_subject.withColumn("subject_id", row_number().over(Window.orderBy("subject")))
    dim_subject.show(5)
    return dim_subject 

In [45]:
dim_subject = dim_subject(work_clean)


====== Create dimension subject  ======


26/05/06 21:18:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+----------+
|        subject|subject_id|
+---------------+----------+
|(Computer file)|         1|
| (Edwin Albert)|         2|
|   (James Yimm)|         3|
|        (Musik)|         4|
|(Robert Edward)|         5|
+---------------+----------+
only showing top 5 rows


26/05/06 21:18:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [46]:
def work_subject(work_clean) :
    print("\n" + "="*60 )
    print("====== Create bridghe work subject table  ======")
    print("="*60 )
    # Tạo bảng bridge work_subject
    work_subject = work_clean.join(dim_subject , on = "subject", how = "inner") \
                    .select("work_id", "subject_id") \
                    .orderBy("subject")
    work_subject.show(10)
    return work_subject

In [47]:
work_subject = work_subject(work_clean)


====== Create bridghe work subject table  ======


26/05/06 21:18:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 2

+-----------+----------+
|    work_id|subject_id|
+-----------+----------+
| OL2582567W|         1|
| OL8887370W|         1|
| OL4327793W|         2|
| OL3018177W|         3|
|   OL28809W|         4|
|   OL28809W|         4|
| OL1863304W|         5|
| OL2186884W|         6|
|OL19447928W|         7|
| OL1840019W|         8|
+-----------+----------+
only showing top 10 rows


In [48]:
def work_author(work_clean):
    print("\n" + "="*60 )
    print("====== Create bridghe work author table  ======")
    print("="*60 )
    # Tạo author schema
    author_schema = ArrayType(
        StructType([
            StructField("key", StringType(), True),
            StructField("name", StringType(), True)
        ])
    )
    # Trích ra cột id và name của author
    work_author = work_clean.withColumn("authors" , from_json(col("authors"), author_schema)) \
                    .withColumn("authors", explode(col("authors"))) \
                    .select("work_id","authors.key")
    # Làm sạch cột author_id
    work_author = work_author.withColumn("key", regexp_extract(col("key"), r"[A-Z]+\d+[A-Z]", 0))
    work_author = work_author.select(col("work_id"), col("key").alias("author_id"))
    work_author.show(5)
    return work_author

In [49]:
work_author = work_author(work_clean)


====== Create bridghe work author table  ======


+----------+----------+
|   work_id| author_id|
+----------+----------+
|OL1652426W| OL184781A|
|  OL19870W|  OL24461A|
|OL5361840W|OL1234665A|
|  OL54797W|  OL19597A|
|OL1652426W| OL184781A|
+----------+----------+
only showing top 5 rows


## Edition

In [ ]:
def edition_clean(editions) :
    print("\n" + "="*60 )
    print("====== Starting Clean editions ======")
    print("="*60 )
    editions = editions.select("key", "works", "title", "publish_date", "publishers", "languages", "number_of_pages", "isbn_13")
    # Tạo work schema
    work_schema = ArrayType(
        StructType([
            StructField("key", StringType(), True)
        ])
    )
    # Làm sạch cột works trích xuất ra work_id
    editions_clean_work = editions.withColumn("works", from_json("works", work_schema)) \
                                .withColumn("works", regexp_extract(col("works")[0]["key"], r"[A-Z]+\d+[A-Z]", 0))
    # Chuyển đổi kiểu dữ liệu publish_date và làm sạch cột key (cột edition_id)
    editions_clean = editions_clean_work.withColumn("publish_date", col("publish_date").cast("string")) \
                    .withColumn("key", regexp_extract("key", r"[A-Z]+\d+[A-Z]", 0))
    # Làm sạch cột publish_date
    publish_date_clean = editions_clean.withColumn("publish_date", trim(regexp_extract("publish_date", r"(\d{4})", 1))) \
                                        .withColumn("publish_date", when(col("publish_date") == "", None)
                                                                    .otherwise(col("publish_date").cast("int"))) 
    # Lấy trung vị cột publish_date
    publish_date_median = publish_date_clean.approxQuantile("publish_date", [0.5], 0.01)[0]

    # Lấy trung vị cột number_of_pages
    number_of_pages_median = publish_date_clean.approxQuantile("number_of_pages", [0.5], 0.01)[0]

    # Làm sạch giá trị null các cột
    editions_clean = publish_date_clean.fillna({"publish_date" : publish_date_median, "number_of_pages" : number_of_pages_median,
                                                "publishers" : "unknown", "languages" : "unknown", "title" : "unknown"})
    editions_clean.select([count(when (col(c).isNull() , c)).alias(c) for c in editions.columns ]).show()
    # Làm sạch cột title
    editions_clean = editions_clean.filter(col("title").rlike("[A-Za-z]")) \
                                    .withColumnRenamed("key", "edition_id") \
                                    .withColumnRenamed("works", "work_id")
    print("Edition count : " , editions_clean.count())
    editions_clean.show(5)
    print("====== Clean editions sucessfully ======")
    return editions_clean

In [51]:
edition_clean = edition_clean(editions)


====== Starting Clean editions ======


+---+-----+-----+------------+----------+---------+---------------+-------+
|key|works|title|publish_date|publishers|languages|number_of_pages|isbn_13|
+---+-----+-----+------------+----------+---------+---------------+-------+
|  0|    0|    0|           0|         0|        0|              0|  59642|
+---+-----+-----+------------+----------+---------+---------------+-------+

Edition count :  223338
+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
| edition_id| work_id|               title|publish_date|          publishers|           languages|number_of_pages|          isbn_13|
+-----------+--------+--------------------+------------+--------------------+--------------------+---------------+-----------------+
|OL61212055M|OL21177W|O Morro dos Vento...|        2014|   ["Martin Claret"]|[{"key": "/langua...|            524|["9788544000182"]|
|OL56513356M|OL21177W| Hauts de Hurle-Vent|        2017|["Creat

In [52]:
def dim_edition(edition_clean) :
    print("\n" + "="*60 )
    print("====== Create dimension edition ======")
    print("="*60 )
    # Tạo bảng dim_editions
    dim_edition = edition_clean.select("edition_id", "title", "publish_date", "publishers", "languages").dropDuplicates() 
    # Explode cột publishers
    dim_edition_clean = dim_edition.withColumn("publishers", from_json("publishers", ArrayType(StringType()))) \
                                    .withColumn("publishers", explode("publishers"))
    # Xóa các khoảng trắng, lọc các dữ liệu bẩn cột publishers
    dim_edition_clean = dim_edition_clean.withColumn("publishers", trim(regexp_replace("publishers", r".$|-$|\?|\]|\[|s\.n\.?|etc\.?|et al\.?|publisher not identified", ""))) \
                                        .filter( (col("publishers").rlike("[a-z]{3,}"))
                                                & (col("publishers") != "")
                                                & (length("publishers") < 86)
                                                & (~col("publishers").rlike(r"printed by|printed for|distributed by|distributor|sold by")))
    # Tạo schema cho cột languages
    language_schema = ArrayType(
        StructType([
            StructField("key", StringType(), True)
        ])
    )
    # Trích xuất ra language từ cột languages
    dim_edition_clean = dim_edition_clean.withColumn("languages", from_json("languages", language_schema)) \
                                        .withColumn("languages" , regexp_extract(col("languages")[0]["key"], "[A-Z|a-z]+$", 0))
    # Đổi các giá trị null sang unknown
    dim_edition_clean = dim_edition_clean.fillna({"languages" : "unknown"})
    # Đổi tên cột publishers, languages
    dim_edition_clean = dim_edition_clean.withColumnRenamed("publishers" , "publisher") \
                                        .withColumnRenamed("languages" , "language")
    language_map = {
        "eng": "English",
        "vie": "Vietnamese",
        "fre": "French",
        "ger": "German",
        "spa": "Spanish",
        "ita": "Italian",
        "jpn": "Japanese",
        "kor": "Korean",
        "chi": "Chinese",
        "rus": "Russian",
        "por": "Portuguese",
        "ara": "Arabic",
        "hin": "Hindi",
        "ben": "Bengali",
        "tur": "Turkish",
        "pol": "Polish",
        "ukr": "Ukrainian",
        "unknown" : "Unknown"
    }
    from itertools import chain
    # Tạo mapping để map cột language sang ngôn ngữ chi tiết
    mapping = create_map(
        [lit(x) for x in chain(*language_map.items())]
    )
    # Mapping cột language
    dim_edition = dim_edition_clean.withColumn("language", mapping[col("language")])
    dim_edition.show(5)
    return dim_edition

In [53]:
dim_edition = dim_edition(edition_clean)


====== Create dimension edition ======


+-----------+--------------------+------------+--------------------+----------+
| edition_id|               title|publish_date|           publisher|  language|
+-----------+--------------------+------------+--------------------+----------+
|OL61212055M|O Morro dos Vento...|        2014|        Martin Clare|Portuguese|
|OL38138835M|Room with a View ...|        2021|Independently Pub...|   English|
|OL55521569M|    Room with a View|        2016|CreateSpace Indep...|   English|
|OL55738986M|    Room with a View|        2016|CreateSpace Indep...|   English|
|OL54816473M|    Room with a View|        2014|CreateSpace Indep...|   English|
+-----------+--------------------+------------+--------------------+----------+
only showing top 5 rows


In [54]:
def dim_time(edition_clean) :
    print("\n" + "="*60 )
    print("====== Create dimension time ======")
    print("="*60 )
    dim_time = edition_clean.select(col("publish_date").alias("publish_year")).distinct()
    # Tạo ra 2 cột decade và century
    dim_time = dim_time.withColumn("decade" , (col("publish_year")/10).cast("int") * 10) \
            .withColumn("century", ((col("publish_year") - 1)/100).cast("int") + 1)
    # Gen ra time_id cho dim_time
    dim_time = dim_time.filter(col("publish_year") >= 1000) \
                        .withColumn("time_id" , row_number().over(Window.orderBy("publish_year"))) \
                        .select("time_id", "publish_year", "decade", "century")
    dim_time.show(5)
    return dim_time

In [55]:
dim_time = dim_time(edition_clean)

26/05/06 21:18:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



====== Create dimension time ======
+-------+------------+------+-------+
|time_id|publish_year|decade|century|
+-------+------------+------+-------+
|      1|        1000|  1000|     10|
|      2|        1200|  1200|     12|
|      3|        1400|  1400|     14|
|      4|        1455|  1450|     15|
|      5|        1465|  1460|     15|
+-------+------------+------+-------+
only showing top 5 rows


26/05/06 21:18:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 21:18:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [85]:
def fact_book(edition_clean):
    print("\n" + "="*60 )
    print("====== Create fact book ======")
    print("="*60 )
    # Tạo bảng fact_book
    fact_book = edition_clean.select("edition_id", "work_id", col("publish_date").alias("publish_year"), "number_of_pages")
    # Join với bảng dim_time lấy time_id
    fact_book = fact_book.join(dim_time.select("time_id", "publish_year"), on = "publish_year", how = "left") \
                        .drop("publish_year") \
                        .select("edition_id", "work_id", "time_id", "number_of_pages")
    fact_book.show(5)
    return fact_book

In [86]:
fact_book = fact_book(edition_clean)


====== Create fact book ======


26/05/06 22:04:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 22:04:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+--------+-------+---------------+
| edition_id| work_id|time_id|number_of_pages|
+-----------+--------+-------+---------------+
|OL61212055M|OL21177W|    514|            524|
|OL56513356M|OL21177W|    517|            356|
|OL56523458M|OL21177W|    517|            368|
|OL56296338M|OL21177W|    517|            356|
|OL55977620M|OL21177W|    516|            150|
+-----------+--------+-------+---------------+
only showing top 5 rows


26/05/06 22:04:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 22:04:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 22:04:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 22:04:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


# Author


In [ ]:
def author_clean(authors) :
    print("\n" + "="*60 )
    print("====== Starting Clean authors ======")
    print("="*60 )
    # Làm sạch cột author_id
    authors_key = authors.withColumn("key", trim(regexp_replace("key", r"/authors/", ""))) \
                        .withColumnRenamed("key", "author_id")
    # Làm sạch cột name
    authors_name = authors_key.filter((col("name").isNotNull()) & (~col("name").rlike("n/a")))
    # Làm sạch cột birth_date 
    author_clean = authors_name.withColumn("birth_date", trim(regexp_extract("birth_date", r"\d{4}", 0)).cast("int")) \
                                .withColumn("birth_date", (rand() * 200 + 1800).cast("int"))
    print("====== Clean authors sucessfully ======")
    author_clean.show(5)
    return author_clean

In [92]:
def dim_author(author_clean) :
    print("====== Create author dimension ======")
    # Tạo bảng dim_author
    dim_author = author_clean.select("author_id", 
                             "name", 
                             "birth_date").dropDuplicates()
    dim_author.show(5)
    return dim_author